# 04 · Predictions and accuracy

This notebook applies the ten models to the ECGs of the MIMIC-IV-ECG test split and measures their accuracy against chronological
age in MIMIC-IV-ECG, CODE-15% and PTB-XL.

1. **Amplitude scale of the Lima et al. model**: the amplitude ratio between CODE-15% tracings and tracings in mV
   (`CODE_AMPLITUDE_RATIO`, notebook 00), and the choice of the scale by the MAE in PTB-XL.
2. **Check of the Lima et al. model** against the predictions its authors released for CODE-15%.
3. **AI-ECG age** (and the logit of male sex) with the ten models in the MIMIC-IV-ECG test split, the CODE-15% test split and
   PTB-XL.
4. **Accuracy**: the MAE in the three datasets, each patient weighted equally (Supplementary Table 2).

| Output | Content |
|---|---|
| `PREDICTIONS/{key}.parquet` | idx, pred_age (years) and sex_logit (not for `lima`): the included ECGs of the MIMIC-IV-ECG test split |
| `PREDICTIONS/external/{key}_{code15, ptbxl}.parquet` | row, pred_age, sex_logit, flat: the CODE-15% test split and PTB-XL (fine-tuned models) |
| `PREDICTIONS/external/lima_ptbxl.parquet` | row, pred_age, flat: PTB-XL with the Lima et al. model |
| `SUMMARY/lima_amplitude_scale.json` | amplitude ratio by lead, the candidate scales and their accuracy in PTB-XL |
| `SUMMARY/accuracy.csv` | key, mae_mimic_test, mae_code15_test, mae_ptbxl (years; NaN where a model was not evaluated), with counts |

Every model is applied to the included ECGs of the MIMIC-IV-ECG test split, which all analyses use (notebooks 05 and 06).

**Needs**: notebooks 01–03 (arrays, cohort, embeddings and trained models in `TRAINED`); the raw PTB-XL and CODE-15% files
(section 1); one CUDA graphics card. **Run time** on one RTX 5090: the Lima et al. model about 7 min for the 372,678 test-split
ECGs, each fine-tuned model about 5–8 min, the frozen models 1–2 min each on the CPU, section 1 a few minutes (reading 21,275
PTB-XL records). Every step skips its output when it exists.

**Credentialed data.** Predictions of models trained on MIMIC-IV labels, and every accuracy in MIMIC-IV-ECG (which uses ages from
MIMIC-IV), stay in `MASTER`. Only aggregate numbers are printed.

In [ ]:
%run ./00_setup_and_settings.ipynb

## What to run

`QUICK_TEST = True` applies every model to the first 256 records of each selection (and the amplitude scale to the 1,000 PTB-XL
records that define it) and writes only below `QUICK_DIR`.

In [ ]:
# ============================== WHAT TO RUN ==============================
QUICK_TEST = False
QUICK_DIR = MASTER / "_quick"                       # outputs of the quick test
QUICK_N = 256
RUN_KEYS = list(MODEL_KEYS)                         # models to apply (all ten)

PRED_DIR = QUICK_DIR / "predictions" if QUICK_TEST else PREDICTIONS
SUMMARY_DIR = QUICK_DIR / "summary" if QUICK_TEST else SUMMARY
for _d in (PRED_DIR, PRED_DIR / "external", SUMMARY_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("quick test" if QUICK_TEST else "full run", "| predictions ->", PRED_DIR, "| summary ->", SUMMARY_DIR)

In [ ]:
import json
import time

import h5py
import torch
from scipy.signal import butter, resample_poly, sosfiltfilt


def first(rows):
    """All rows, or the first QUICK_N of them in the quick test."""
    rows = np.sort(np.asarray(rows))
    return rows[:QUICK_N] if QUICK_TEST else rows

## 1. Amplitude scale of the Lima et al. model

The Lima et al. model was trained on CODE ECGs, whose amplitude units differ from millivolts: the tracings of CODE-15% are about
1.8 times larger than tracings in mV. The ratio is computed as follows. For each lead, the amplitude of a record is its robust
range (99.5th minus 0.5th percentile after a 0.5-Hz high-pass filter, second-order Butterworth, applied forwards and backwards);
the median of this range over 500 CODE-15% exams (the first 1,500 exams of `exams_part17.hdf5`, keeping those with more than 2,000
non-zero samples, and within each exam only the samples where some lead is non-zero) is divided by its median over the first
1,000 PTB-XL records at ages of 18–88 years. The ratio is the median of the 12 lead ratios.

The model is then applied to all PTB-XL records at ages of 18–88 years (500 Hz, mV, standard lead order, resampled to 400 Hz and
centred in 4,096 samples), once with the unscaled signal (the millivolt scale of the authors' instructions) and once with the signal
multiplied by the ratio. The scale with the lower MAE (each patient weighted equally) is chosen, so PTB-XL is not an independent
evaluation of this model.

In [ ]:
SCALE_CODE15_FILE = CODE15_DIR / "exams_part17.hdf5"
SCALE_CODE15_EXAMS, SCALE_CODE15_USED, SCALE_PTBXL_USED = 1500, 500, 1000


def robust_range(x, fs):
    s = sosfiltfilt(butter(2, 0.5, "highpass", fs=fs, output="sos"), x, axis=-1)
    return np.percentile(s, 99.5, axis=-1) - np.percentile(s, 0.5, axis=-1)


def read_ptbxl_mv(filenames):
    """PTB-XL records at 500 Hz in mV as (n, 12, 5000) float32 in standard lead order; missing values set to 0."""
    import wfdb
    return np.nan_to_num(np.stack([wfdb.rdsamp(str(PTBXL_DIR / f))[0].T for f in filenames]).astype(np.float32))


def amplitude_ratio(db):
    with h5py.File(SCALE_CODE15_FILE, "r") as f:
        cx = f["tracings"][:SCALE_CODE15_EXAMS].transpose(0, 2, 1)          # (exams, 12, 4096), 400 Hz
    keep = (np.abs(cx).sum(axis=1) > 0).sum(axis=1) > 2000
    ptb = read_ptbxl_mv(db["filename_hr"].iloc[:SCALE_PTBXL_USED])
    by_lead = []
    for k in range(12):
        c = np.median([robust_range(r[k][np.abs(r).sum(0) > 0], 400) for r in cx[keep][:SCALE_CODE15_USED]])
        p = np.median(robust_range(ptb[:, k], 500))
        by_lead.append(float(c / p))
    return float(np.median(by_lead)), by_lead


def lima_ptbxl(db, scales, chunk=2048):
    """Predictions of the Lima et al. model for the PTB-XL records of `db` at each amplitude scale (batches of 256, float32)."""
    dev = gpu_device()
    gpu_precision(tf32_matmul=False, tf32_conv=True, benchmark=False)
    model = lima_model().to(dev)
    preds, flat = {name: [] for name in scales}, []
    with torch.no_grad():
        for s in range(0, len(db), chunk):
            sig = read_ptbxl_mv(db["filename_hr"].iloc[s:s + chunk])
            flat.append(sig.reshape(len(sig), -1).max(1) == sig.reshape(len(sig), -1).min(1))
            x400 = np.pad(resample_poly(sig, 4, 5, axis=-1).astype(np.float32), ((0, 0), (0, 0), (48, 48)))
            for name, sc in scales.items():
                for b in range(0, len(x400), 256):
                    preds[name].append(model(torch.from_numpy(x400[b:b + 256] * sc).to(dev)).cpu().numpy().ravel())
    del model
    torch.cuda.empty_cache()
    return {name: np.concatenate(p) for name, p in preds.items()}, np.concatenate(flat)


scale_json = SUMMARY_DIR / "lima_amplitude_scale.json"
lima_ptb_path = PRED_DIR / "external" / "lima_ptbxl.parquet"
if "lima" in RUN_KEYS and not (scale_json.exists() and lima_ptb_path.exists()):     # a few minutes (reading PTB-XL)
    t0 = time.time()
    db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv")
    db = db[(db["age"] >= AGE_MIN) & (db["age"] < AGE_MAX + 1)].reset_index(drop=True)
    ratio, by_lead = amplitude_ratio(db)
    if QUICK_TEST:
        db = db.head(SCALE_PTBXL_USED)
    scales = {"mV": 1.0, "CODE-matched": ratio}
    preds, flat = lima_ptbxl(db, scales)
    pid = db["patient_id"].to_numpy()
    res = {}
    for name, sc in scales.items():
        e = preds[name] - db["age"].to_numpy()
        res[name] = {"scale": sc, "mae": float(np.average(np.abs(e), weights=patient_weights(pid)))}
    chosen = min(res, key=lambda k: res[k]["mae"])
    out = {"ratio_by_lead": by_lead, "ratio": ratio, "candidates": res, "chosen": chosen, "chosen_scale": res[chosen]["scale"],
           "n_ecg": int(len(db)), "n_patients": int(len(np.unique(pid))), "ptbxl_records_for_ratio": SCALE_PTBXL_USED,
           "code15_exams_for_ratio": SCALE_CODE15_USED}
    rows = pd.read_parquet(PTBXL_RECORDS, columns=["row", "ecg_id"]).set_index("ecg_id")["row"].reindex(db["ecg_id"]).to_numpy()
    pd.DataFrame({"row": rows.astype(np.int64), "pred_age": preds[chosen].astype(np.float32), "flat": flat}).sort_values("row") \
        .to_parquet(lima_ptb_path, index=False)
    scale_json.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print(f"amplitude ratio CODE-15% / PTB-XL by lead: {[round(r, 2) for r in by_lead]} -> median {ratio:.4f}")
    for name, r in res.items():
        print(f"  {name:13s} scale {r['scale']:.3f}: MAE {r['mae']:.2f} y")
    print(f"chosen: {chosen}; {(time.time() - t0) / 60:.1f} min")
elif scale_json.exists():
    print("amplitude scale exists:", json.loads(scale_json.read_text())["chosen"])

## 2. Check: the Lima et al. model against its authors' CODE-15% predictions

CODE-15% releases, for every exam, the age predicted by the authors with this model (`nn_predicted_age` in `exams.csv`). This cell
feeds the first 3,000 tracings of `exams_part17.hdf5` to the model as they are in the file (400 Hz, 4,096 samples, CODE amplitude
units), as the authors' `evaluate.py` does, and compares the predicted ages with `nn_predicted_age`: mean and maximum absolute
difference and correlation, written as `code15_authors_check` to `SUMMARY/lima_amplitude_scale.json`. The comparison concerns the
network and the weights; the conversion of 500-Hz millivolt ECGs is assessed by the PTB-XL accuracy in section 1.

In [ ]:
if "lima" in RUN_KEYS:                              # about 1 min
    out = json.loads(scale_json.read_text()) if scale_json.exists() else {}
    if "code15_authors_check" not in out:
        dev = gpu_device()
        gpu_precision(tf32_matmul=False, tf32_conv=True, benchmark=False)
        model = lima_model().to(dev)
        with h5py.File(SCALE_CODE15_FILE, "r") as f:
            ids = f["exam_id"][:]
            n = min(len(ids), QUICK_N if QUICK_TEST else 3000)
            x = f["tracings"][:n]                     # (n, 4096, 12)
        with torch.no_grad():
            p = np.concatenate([model(torch.tensor(x[s:s + 64]).transpose(-1, -2).to(dev, dtype=torch.float32)).cpu().numpy().ravel()
                                for s in range(0, n, 64)])
        ref = pd.Series(ids[:n]).map(pd.read_csv(CODE15_DIR / "exams.csv", usecols=["exam_id", "nn_predicted_age"])
                                     .set_index("exam_id")["nn_predicted_age"]).to_numpy()
        ok = ~np.isnan(ref)
        out["code15_authors_check"] = {"exams": int(ok.sum()), "mean_abs_difference": float(np.abs(p[ok] - ref[ok]).mean()),
                                       "max_abs_difference": float(np.abs(p[ok] - ref[ok]).max()),
                                       "r": float(np.corrcoef(p[ok], ref[ok])[0, 1])}
        scale_json.write_text(json.dumps(out, indent=2), encoding="utf-8")
        del model
        torch.cuda.empty_cache()
    c = out["code15_authors_check"]
    print(f"Lima et al. model against the authors' predictions in {c['exams']:,} CODE-15% exams: mean absolute difference "
          f"{c['mean_abs_difference']:.4f} y, maximum {c['max_abs_difference']:.3f} y, r {c['r']:.5f}")

## 3. AI-ECG age of the test-split ECGs

### 3.1 The Lima et al. model

The included ECGs of the test split, prepared as in section 1 with the scale `CODE_AMPLITUDE_RATIO`, 256 records per batch, in
float32. (Section 1 writes the PTB-XL predictions.)

In [ ]:
def predict_lima_mimic(path):
    t0 = time.time()
    dev = gpu_device()
    gpu_precision(tf32_matmul=False, tf32_conv=True, benchmark=False)
    model = lima_model().to(dev)
    arr = np.load(MIMIC_WAVE500, mmap_mode="r")
    lab = model_labels("mimic")
    rows = first(lab.index[lab["split"] == "test"])
    out_r, out_a = [], []
    with torch.no_grad():
        for r, x in block_batches(arr, rows, 256, block=4096):
            out_r.append(r)
            out_a.append(model(torch.from_numpy(lima_input(x, CODE_AMPLITUDE_RATIO)).to(dev)).float().cpu().numpy().ravel())
    pd.DataFrame({"idx": np.concatenate(out_r).astype(np.int64), "pred_age": np.concatenate(out_a).astype(np.float32)}) \
        .to_parquet(path, index=False)
    del model
    torch.cuda.empty_cache()
    print(f"lima: {len(rows):,} MIMIC-IV-ECG records in {(time.time() - t0) / 60:.1f} min")


if "lima" in RUN_KEYS:                              # about 7 min for the 372,678 test-split ECGs
    path = PRED_DIR / "lima.parquet"
    if path.exists():
        print("lima: predictions exist, skipped")
    else:
        predict_lima_mimic(path)

### 3.2 The frozen models

The probe of notebook 03 (`TRAINED/{key}/ridge.npz`) applied to the frozen embeddings of the included ECGs of the test split.

In [ ]:
def predict_frozen(key, path):
    fm = model_parts(key)[0]
    E = np.load(EMBEDDINGS / f"{fm}_f16.npy", mmap_mode="r")
    probe = dict(np.load(TRAINED / key / "ridge.npz"))
    lab = model_labels("mimic")
    lab = lab[(lab.index < E.shape[0]) & (lab["split"] == "test")]
    rows = first(lab.index.to_numpy())
    age, logit = [], []
    for s in range(0, len(rows), 100_000):
        a, l = probe_predict(E[rows[s:s + 100_000]], probe)
        age.append(a)
        logit.append(l)
    pd.DataFrame({"idx": rows.astype(np.int64),
                  "pred_age": np.concatenate(age).astype(np.float32), "sex_logit": np.concatenate(logit).astype(np.float32)}) \
        .to_parquet(path, index=False)
    print(f"{key}: {len(rows):,} MIMIC-IV-ECG records (ridge penalty {float(probe['ridge_alpha']):g}, "
          f"logistic C {float(probe['logistic_C']):g})")


for key in [k for k in RUN_KEYS if k.endswith("_frozen")]:      # 1-2 min each on the CPU
    path = PRED_DIR / f"{key}.parquet"
    if path.exists():
        print(f"{key}: predictions exist, skipped")
        continue
    predict_frozen(key, path)

### 3.3 The fine-tuned models

Each checkpoint `TRAINED/{key}/best.pt` is loaded with `strict=True` and applied in bfloat16 mixed precision, with the batches of
its training (section 4 of notebook 03), to the included ECGs of the MIMIC-IV-ECG test split, to the CODE-15% test split and to
PTB-XL at ages of 18–88 years. `flat` marks constant records (72 in the CODE-15% test split), which are left out of the accuracy.

In [ ]:
def predict_finetuned(key, mimic_path):
    t0 = time.time()
    fm = model_parts(key)[0]
    cfg = FINETUNE[fm]
    dev = gpu_device()
    gpu_precision(tf32_matmul=True, tf32_conv=True, benchmark=True)
    model, ck = load_trained(key)
    model = model.to(dev).eval()
    arr = model_inputs("mimic", fm)
    lab = model_labels("mimic")
    lab = lab[lab.index < arr.shape[0]]
    r, a, s, _ = predict_records(model, arr, first(lab.index[lab["split"] == "test"]), cfg["bs"], cfg["block"], dev)
    pd.DataFrame({"idx": r.astype(np.int64), "pred_age": a.astype(np.float32),
                  "sex_logit": s.astype(np.float32)}).to_parquet(mimic_path, index=False)
    msg = [f"MIMIC-IV-ECG {len(r):,}"]
    for src in ("code15", "ptbxl"):
        ea, el = model_inputs(src, fm), model_labels(src)
        el = el[el.index < ea.shape[0]]
        r, a, s, fl = predict_records(model, ea, first(el.index[el["split"] == "test"]), cfg["bs"], cfg["block"], dev)
        pd.DataFrame({"row": r.astype(np.int64), "pred_age": a.astype(np.float32), "sex_logit": s.astype(np.float32), "flat": fl}) \
            .to_parquet(PRED_DIR / "external" / f"{key}_{src}.parquet", index=False)
        msg.append(f"{'CODE-15% test' if src == 'code15' else 'PTB-XL'} {len(r):,} ({int(fl.sum())} flat)")
    del model
    torch.cuda.empty_cache()
    print(f"{key} (step {ck['step']:,}): {', '.join(msg)} records in {(time.time() - t0) / 60:.1f} min")


for key in [k for k in RUN_KEYS if model_parts(k)[1] in ("mimic", "code15")]:      # about 5-8 min each on one RTX 5090
    mimic_path = PRED_DIR / f"{key}.parquet"
    ext = [PRED_DIR / "external" / f"{key}_{src}.parquet" for src in ("code15", "ptbxl")]
    if mimic_path.exists() and all(p.exists() for p in ext):
        print(f"{key}: predictions exist, skipped")
        continue
    predict_finetuned(key, mimic_path)

## 4. Accuracy (Supplementary Table 2)

Mean absolute error of AI-ECG age against chronological age in the MIMIC-IV-ECG test split (included ECGs), the CODE-15% test
split and PTB-XL (ages of 18–88 years; flat records left out), each ECG weighted by 1 / (number of ECGs of its patient), so that
every patient counts equally. Models that were not applied to a dataset have NaN.

In [ ]:
def accuracy_table(pred_dir):
    test = model_labels("mimic")
    test = test[test["split"] == "test"]
    ext_labels = {src: model_labels(src) for src in ("code15", "ptbxl")}
    out = []
    for key in MODEL_KEYS:
        path = pred_dir / f"{key}.parquet"
        row = {"key": key}
        if path.exists():
            p = pd.read_parquet(path, columns=["idx", "pred_age"]).set_index("idx")["pred_age"]
            sel = test.index.intersection(p.index[p.notna()])
            m = age_metrics(test.loc[sel, "age"], p.loc[sel], test.loc[sel, "pid"])
            row.update(mae_mimic_test=m.get("mae"), n_ecg_mimic_test=m["n_age"], n_patients_mimic_test=m.get("n_patients"))
        for src, col in (("code15", "code15_test"), ("ptbxl", "ptbxl")):
            path = pred_dir / "external" / f"{key}_{src}.parquet"
            if not path.exists():
                continue
            e = pd.read_parquet(path)
            e = e[~e["flat"]]
            lab = ext_labels[src].loc[e["row"]]
            m = age_metrics(lab["age"], e["pred_age"], lab["pid"])
            row.update({f"mae_{col}": m.get("mae"), f"n_ecg_{col}": m["n_age"]})
        out.append(row)
    cols = ["key", "mae_mimic_test", "mae_code15_test", "mae_ptbxl", "n_ecg_mimic_test", "n_patients_mimic_test", "n_ecg_code15_test",
            "n_ecg_ptbxl"]
    table = pd.DataFrame(out).reindex(columns=cols)
    counts = [c for c in cols if c.startswith("n_")]
    table[counts] = table[counts].astype("Int64")              # counts stay integers where a model was not evaluated
    return table


acc = accuracy_table(PRED_DIR)                      # about 1 min
acc.to_csv(SUMMARY_DIR / "accuracy.csv", index=False)
show = acc[["key", "mae_mimic_test", "mae_code15_test", "mae_ptbxl"]].copy()
show.insert(1, "model", show["key"].map(model_label))
display(show.drop(columns="key").round(2))